<a href="https://colab.research.google.com/github/Vineet-salimath/NLP_Preprocessing_Engine/blob/main/Chatbot_using_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BUILD  A CHATBOT USING HUGGING FACE TRANSFORMERS


Install Libraries

In [11]:

!pip install transformers torch accelerate --quiet

Import the Libraries

In [12]:
import warnings
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import logging

warnings.filterwarnings('ignore')
logging.getLogger("transformers").setLevel(logging.ERROR)

In [13]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"

Model Loading

In [14]:

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Qwen Model loaded successfully!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen Model loaded successfully!


The Generator


In [18]:
def generate_reply(chat_history, user_message, max_new_tokens=200):

    chat_history.append({"role": "user", "content": user_message})

    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )


    inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    assistant_reply = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


    chat_history.append({"role": "assistant", "content": assistant_reply})

    return chat_history, assistant_reply

Chat Interface




In [21]:
def run_chatbot():
    print("Chatbuddy: Hello! I am your AI assistant. How can I help you today?")
    chat_history = [
        {"role": "system", "content": "You are a helpful AI assistant. Answer clearly and politely in a conversational tone."}
    ]

    while True:
        user_input = input("User: ").strip()

        if user_input.lower() in ["exit", "quit"]:
            print(" Chatbuddy ends the conversation")
            break

        if not user_input:
            continue
        chat_history, assistant_reply = generate_reply(chat_history, user_input)
        print(f" Chatbuddy: {assistant_reply}")

run_chatbot()

Chatbuddy: Hello! I am your AI assistant. How can I help you today?
User: hey there Vineet this side
 Chatbuddy: Hello! How can I assist you today?
User: what is artificial intelligence??
 Chatbuddy: Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules to reach approximate or definite conclusions), and self-correction. AI focuses on creating intelligent machines that work and react like humans. It encompasses various subfields such as machine learning, natural language processing, computer vision, robotics, and expert systems.
User: who invented python?
 Chatbuddy: Python was invented by Guido van Rossum in 1989. He started working on it while he was a researcher at the STScI in San Francisco. The first version was called Python 0.9.0. Guido van Rossum is considered the primary creator an